# D140 — MySQL Complete Basics

A practical 90–120 minute tour of MySQL fundamentals using a small order database. Run the cells from top to bottom. The notebook intentionally emphasizes runnable SQL, simple examples, and reusable Python helpers rather than database-design theory.

## Setup and connection

Install the driver once if needed: `pip install mysql-connector-python`. Edit the password below before running. Later, switch to the commented `os.environ` form so credentials are not stored in a notebook.

**InnoDB** is MySQL's default storage engine. It stores and manages table data and supports important features such as transactions, foreign keys, row-level locking, and crash recovery. The tables in this notebook use InnoDB so relationships and data changes are handled reliably.

In [ ]:
import os
import mysql.connector
from mysql.connector import Error

# Learning setup: hard-coded in the FIRST code cell as requested.
# Change MYSQL_PASSWORD to the password for your local MySQL root account.
MYSQL_HOSTNAME = "localhost"
MYSQL_PORT = 3306
MYSQL_USERNAME = "root"
MYSQL_PASSWORD = "root"
MYSQL_DATABASE = "orderdb"

# Recommended later (set these variables in your operating system first):
# MYSQL_HOSTNAME = os.environ["MYSQL_HOSTNAME"]
# MYSQL_PORT = int(os.environ.get("MYSQL_PORT", "3306"))
# MYSQL_USERNAME = os.environ["MYSQL_USERNAME"]
# MYSQL_PASSWORD = os.environ["MYSQL_PASSWORD"]
# MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE", "orderdb")

In [ ]:
# Connect without selecting a database so the notebook can create orderdb.
server_connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
)
server_cursor = server_connection.cursor()

query = f"""
CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}`
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci
"""
server_cursor.execute(query)
server_cursor.close()
server_connection.close()
print(f"Database {MYSQL_DATABASE!r} is ready.")

In [ ]:
connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)
print("Connected:", connection.is_connected())
print("MySQL version:", connection.get_server_info())

### Reusable SQL helpers

A cursor sends SQL to MySQL and reads returned rows. The helper below detects whether a statement produced a result set. It prints query results for `SELECT`, `SHOW`, `DESCRIBE`, and procedure result sets; otherwise it commits and prints affected rows. Parameters must be passed separately—never place untrusted values directly into an f-string.

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return
    print(" | ".join(columns))
    print("-+-".join("-" * len(column) for column in columns))
    for row in rows:
        print(" | ".join(str(value) for value in row))


def execute_sql(sql, params=None, *, many=False):
    """Execute one SQL statement and print either results or affected rows."""
    cursor = connection.cursor()
    try:
        if many:
            cursor.executemany(sql, params or [])
        else:
            cursor.execute(sql, params or ())

        result_sets = []
        affected_rows = cursor.rowcount
        while True:
            if cursor.with_rows:
                columns = [item[0] for item in cursor.description]
                rows = cursor.fetchall()
                print_rows(columns, rows)
                result_sets.append(rows)
            if not cursor.nextset():
                break

        if result_sets:
            return result_sets[0] if len(result_sets) == 1 else result_sets

        connection.commit()
        print(f"Rows impacted: {affected_rows}")
        return affected_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


def execute_script(statements):
    """Run a list of individual SQL statements in order."""
    for number, statement in enumerate(statements, start=1):
        print(f"\nStatement {number}:")
        execute_sql(statement)

## Database, schema, tables, rows, and columns

- A **database** is an organized collection of data. In MySQL, `DATABASE` and `SCHEMA` are synonyms.
- A **table** stores related data. A **row** is one record; a **column** is one named attribute with a data type.
- Prefer lowercase `snake_case`, meaningful singular or plural names used consistently, and avoid spaces/reserved words.
- A **data dictionary** is metadata about database objects. MySQL exposes much of it through `information_schema`.

The reset below makes repeat runs predictable. Child tables are dropped before parent tables because of foreign keys.

### Remove the existing `order_summary` view

This makes the setup repeatable before the view is created again later.

In [ ]:
query = f"""
DROP VIEW IF EXISTS order_summary
"""
execute_sql(query)

### Remove the existing `products_after_update` trigger

Removing the old trigger prevents a duplicate-object error when the notebook is rerun.

In [ ]:
query = f"""
DROP TRIGGER IF EXISTS products_after_update
"""
execute_sql(query)

### Remove the existing `orders_for_customer` procedure

This clears the previous procedure definition before the demonstration recreates it.

In [ ]:
query = f"""
DROP PROCEDURE IF EXISTS orders_for_customer
"""
execute_sql(query)

### Remove the existing `calculate_gst` function

This clears the previous stored function so the notebook remains repeatable.

In [ ]:
query = f"""
DROP FUNCTION IF EXISTS calculate_gst
"""
execute_sql(query)

### Remove the existing `product_audit` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS product_audit
"""
execute_sql(query)

### Remove the existing `order_items` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS order_items
"""
execute_sql(query)

### Remove the existing `orders` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS orders
"""
execute_sql(query)

### Remove the existing `customer_profiles` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS customer_profiles
"""
execute_sql(query)

### Remove the existing `products` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS products
"""
execute_sql(query)

### Remove the existing `customers` table

Tables are removed in dependency order so foreign-key relationships do not block the reset.

In [ ]:
query = f"""
DROP TABLE IF EXISTS customers
"""
execute_sql(query)

### Create the `customers` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE customers (
        customer_id INT AUTO_INCREMENT PRIMARY KEY,
        customer_name VARCHAR(100) NOT NULL,
        email VARCHAR(255) NOT NULL UNIQUE,
        city VARCHAR(80),
        created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
    ) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `customer_profiles` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE customer_profiles (
        customer_id INT PRIMARY KEY,
        phone VARCHAR(20),
        CONSTRAINT fk_profile_customer
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
            ON DELETE CASCADE
    ) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `products` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE products (
        product_id INT AUTO_INCREMENT PRIMARY KEY,
        product_name VARCHAR(100) NOT NULL,
        sku VARCHAR(30) NOT NULL UNIQUE,
        price DECIMAL(10, 2) NOT NULL,
        stock_quantity INT NOT NULL DEFAULT 0,
        is_active BOOLEAN NOT NULL DEFAULT TRUE,
        CONSTRAINT chk_product_price CHECK (price >= 0),
        CONSTRAINT chk_product_stock CHECK (stock_quantity >= 0)
    ) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `orders` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE orders (
        order_id INT AUTO_INCREMENT PRIMARY KEY,
        customer_id INT NOT NULL,
        order_date DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
        status VARCHAR(20) NOT NULL DEFAULT 'NEW',
        CONSTRAINT chk_order_status CHECK (status IN ('NEW', 'PAID', 'SHIPPED', 'CANCELLED')),
        CONSTRAINT fk_order_customer
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    ) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `order_items` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE order_items (
        order_id INT NOT NULL,
        product_id INT NOT NULL,
        quantity INT NOT NULL DEFAULT 1,
        unit_price DECIMAL(10, 2) NOT NULL,
        PRIMARY KEY (order_id, product_id),
        CONSTRAINT chk_item_quantity CHECK (quantity > 0),
        CONSTRAINT fk_item_order FOREIGN KEY (order_id) REFERENCES orders(order_id) ON DELETE CASCADE,
        CONSTRAINT fk_item_product FOREIGN KEY (product_id) REFERENCES products(product_id)
    ) ENGINE=InnoDB
"""
execute_sql(query)

### Constraints and relationships demonstrated

- `PRIMARY KEY`: unique row identity; `AUTO_INCREMENT` generates integer IDs.
- `FOREIGN KEY`: ensures referenced parent rows exist—this is **referential integrity**.
- `UNIQUE`: prevents duplicate email/SKU values. `NOT NULL`: requires a value.
- `DEFAULT`: supplies a value when omitted. `CHECK`: validates a rule (enforced in modern MySQL 8.x).
- One-to-one: `customers` → `customer_profiles`, because the foreign key is also the profile primary key.
- One-to-many: one customer → many orders.
- Many-to-many: orders ↔ products through the junction table `order_items`.

### Insert rows into `customers`

The statement adds sample records that later queries will use.

Use placeholders (%s) for data values. This is safer than f-string interpolation.

In [ ]:
query = f"""
INSERT INTO customers (customer_name, email, city)
VALUES (%s, %s, %s)
"""
customers_data = [
    ("Venkat", "venkat@example.com", "Chennai"),
    ("Bilal", "bilal@example.com", "Hyderabad"),
    ("Chen", "chen@example.com", "Bengaluru"),
]
execute_sql(query, customers_data, many=True)

### Insert rows into `products`

The statement adds sample records that later queries will use.

In [ ]:
query = f"""
INSERT INTO products (product_name, sku, price, stock_quantity)
VALUES (%s, %s, %s, %s)
"""
products_data = [
    ("Notebook", "NB-100", 80.00, 100),
    ("Pen Set", "PN-200", 120.00, 50),
    ("Desk Lamp", "DL-300", 1500.00, 12),
]
execute_sql(query, products_data, many=True)

### Insert rows into `customer_profiles`

The statement adds sample records that later queries will use.

In [ ]:
query = f"""
INSERT INTO customer_profiles (customer_id, phone)
VALUES (1, '9000000001'), (2, '9000000002')
"""
execute_sql(query)

### Insert rows into `orders`

The statement adds sample records that later queries will use.

In [ ]:
query = f"""
INSERT INTO orders (customer_id, status)
VALUES (1, 'PAID'), (1, 'NEW'), (2, 'SHIPPED')
"""
execute_sql(query)

### Insert rows into `order_items`

The statement adds sample records that later queries will use.

In [ ]:
query = f"""
INSERT INTO order_items (order_id, product_id, quantity, unit_price)
VALUES
    (1, 1, 2, 80.00),
    (1, 2, 1, 120.00),
    (2, 3, 1, 1500.00),
    (3, 2, 3, 120.00)
"""
execute_sql(query)

### Select products

Filter and order product rows while returning only the columns needed by the result.

Basic CRUD: SELECT, UPDATE, and DELETE.

In [ ]:
query = f"""
SELECT product_id, product_name, price, stock_quantity
FROM products
WHERE price >= %s
ORDER BY price DESC
"""
execute_sql(query, (100,))

### Update rows in `products`

The `WHERE` condition controls which existing rows are changed.

In [ ]:
query = f"""
UPDATE products
SET stock_quantity = stock_quantity + %s
WHERE sku = %s
"""
execute_sql(query, (10, "NB-100"))

# Delete example deliberately affects zero rows.

### Delete rows from `products`

This safe demonstration uses a value that does not exist, so it reports zero affected rows.

In [ ]:
query = f"""
DELETE FROM products WHERE sku = %s
"""
execute_sql(query, ("DOES-NOT-EXIST",))

## Joins and relationship queries

SQL is set-based: this one query combines all matching rows and calculates totals without manually looping through orders.

In [ ]:
query = f"""
SELECT
    o.order_id,
    c.customer_name,
    o.status,
    SUM(oi.quantity * oi.unit_price) AS order_total
FROM orders AS o
JOIN customers AS c ON c.customer_id = o.customer_id
JOIN order_items AS oi ON oi.order_id = o.order_id
GROUP BY o.order_id, c.customer_name, o.status
ORDER BY o.order_id
"""
execute_sql(query)

## Database objects

### Views
A view is a saved query that can simplify access, hide complexity, and expose only selected columns. A simple single-table view may be updatable; views with aggregation, grouping, or many joins are generally read-only. Permissions on views can also provide a security boundary, but secure design still requires careful grants and testing.

### Create the `order_summary` view

The view saves a reusable query and hides the details of its joins and aggregation.

In [ ]:
query = f"""
CREATE OR REPLACE VIEW order_summary AS
SELECT
    o.order_id,
    c.customer_name,
    o.order_date,
    o.status,
    SUM(oi.quantity * oi.unit_price) AS subtotal
FROM orders AS o
JOIN customers AS c ON c.customer_id = o.customer_id
JOIN order_items AS oi ON oi.order_id = o.order_id
GROUP BY o.order_id, c.customer_name, o.order_date, o.status
"""
execute_sql(query)

### Query the order-summary view

Read the view like a table without repeating its underlying joins and aggregation.

In [ ]:
query = f"""
SELECT * FROM order_summary ORDER BY order_id
"""
execute_sql(query)

### Stored functions and procedures

A **function** returns one value and can appear inside an expression, making it useful for calculations or formatting. A **procedure** is called as a command, can return result sets, and may use `IN`, `OUT`, or `INOUT` parameters. Enterprises sometimes centralize shared calculations, controlled data operations, or batch work in these objects. Too much database-side logic can be harder to test and deploy, so use it deliberately.

Connector calls do not need the command-line client's `DELIMITER` directive; each definition is sent as one complete string. Creating routines may require suitable privileges.

### Create the `calculate_gst` stored function

This database function returns a single calculated value that SQL expressions can reuse.

In [ ]:
query = f"""
CREATE FUNCTION calculate_gst(amount DECIMAL(12, 2), gst_rate DECIMAL(5, 2))
RETURNS DECIMAL(12, 2)
DETERMINISTIC
RETURN ROUND(amount * gst_rate / 100, 2)
"""
execute_sql(query)

### Calculate GST for each order

Use the stored function inside a `SELECT` expression to calculate GST and the final order amount.

In [ ]:
query = f"""
SELECT
    order_id,
    subtotal,
    calculate_gst(subtotal, 18.00) AS gst,
    subtotal + calculate_gst(subtotal, 18.00) AS total_with_gst
FROM order_summary
ORDER BY order_id
"""
execute_sql(query)

### Create the `orders_for_customer` stored procedure

The procedure accepts an input parameter and returns a customer-specific result set.

In [ ]:
query = f"""
CREATE PROCEDURE orders_for_customer(IN requested_customer_id INT)
BEGIN
    SELECT order_id, order_date, status
    FROM orders
    WHERE customer_id = requested_customer_id
    ORDER BY order_date;
END
"""
execute_sql(query)

### Call the `orders_for_customer` procedure

Pass an input value and inspect the result set returned by the procedure.

In [ ]:
query = f"""
CALL orders_for_customer(%s)
"""
execute_sql(query, (1,))

# OUT concept: a procedure can assign SET output_parameter = ...; Python can
# retrieve it with cursor.callproc(...). This lab keeps the runnable example simple.

### Triggers
A trigger runs automatically `BEFORE` or `AFTER` an `INSERT`, `UPDATE`, or `DELETE`. `BEFORE` is useful for validation or adjusting `NEW` values. `AFTER` is useful for audit rows after a change succeeds. Triggers can hide behavior, so document and keep them small. This example records price changes after an update.

### Create the `product_audit` table

Review its columns, data types, keys, defaults, and constraints before executing the statement.

In [ ]:
query = f"""
CREATE TABLE product_audit (
    audit_id INT AUTO_INCREMENT PRIMARY KEY,
    product_id INT NOT NULL,
    old_price DECIMAL(10, 2) NOT NULL,
    new_price DECIMAL(10, 2) NOT NULL,
    changed_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `products_after_update` trigger

The trigger runs automatically when a product row is updated.

In [ ]:
query = f"""
CREATE TRIGGER products_after_update
AFTER UPDATE ON products
FOR EACH ROW
BEGIN
    IF NEW.price <> OLD.price THEN
        INSERT INTO product_audit (product_id, old_price, new_price)
        VALUES (OLD.product_id, OLD.price, NEW.price);
    END IF;
END
"""
execute_sql(query)

### Update rows in `products`

The `WHERE` condition controls which existing rows are changed.

In [ ]:
query = f"""
UPDATE products SET price = 85.00 WHERE sku = 'NB-100'
"""
execute_sql(query)

### Run the next SQL statement

Read the SQL first, predict its result, and then execute it with the reusable helper.

In [ ]:
query = f"""
SELECT * FROM product_audit ORDER BY audit_id
"""
execute_sql(query)

## Cursor concepts

The word **cursor** has two related uses here:

1. A Python connector cursor sends statements and fetches results. We used one inside `execute_sql`.
2. A stored-program cursor lets MySQL process a result one row at a time.

SQL is designed for set operations. Prefer one `UPDATE`, join, aggregate, window function, or common table expression over a row-by-row loop. Stored cursors can be appropriate when every row requires ordered procedural work that cannot be expressed clearly as a set operation. They usually add code and network/engine overhead, hold resources longer, and scale poorly.

Compare the following ideas:

```sql
-- Row-by-row idea: fetch each low-stock product, then update it.
-- Set-based alternative (preferred):
UPDATE products
SET stock_quantity = stock_quantity + 10
WHERE stock_quantity < 20;
```

Stored cursor syntax involves `DECLARE CURSOR`, a `NOT FOUND` handler, `OPEN`, repeated `FETCH`, and `CLOSE`. Knowing when *not* to use it is more important in this overview.

In [ ]:
query = f"""
UPDATE products
SET stock_quantity = stock_quantity + 10
WHERE stock_quantity < 20
"""
execute_sql(query)

## Temporary tables

A temporary table is visible only to the current database session and disappears when that connection closes. It is useful for staging intermediate results, simplifying a multi-step transformation, or repeatedly querying a small derived result. It can still consume memory/disk and often a CTE or derived table is simpler.

### Create the temporary `customer_spend` table

This materializes an intermediate result for the current MySQL session only.

In [ ]:
query = f"""
CREATE TEMPORARY TABLE customer_spend AS
SELECT c.customer_id, c.customer_name, COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS total_spend
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
LEFT JOIN order_items AS oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.customer_name
"""
execute_sql(query)

### Query the temporary result

Read the session-scoped temporary table and rank customers by total spending.

In [ ]:
query = f"""
SELECT * FROM customer_spend ORDER BY total_spend DESC
"""
execute_sql(query)

## Metadata and the system catalog

`information_schema` is MySQL's portable-style metadata database. `SHOW` and `DESCRIBE` are convenient MySQL shortcuts. Metadata lets tools inspect tables, columns, constraints, and indexes without reading application data.

### Inspect database objects

List the base tables and views registered in the `orderdb` schema.

In [ ]:
query = f"""
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = %s
ORDER BY table_type, table_name
"""
execute_sql(query, (MYSQL_DATABASE,))

### Inspect product columns

Read column names, types, nullability, defaults, and key information from the data dictionary.

In [ ]:
query = f"""
SELECT column_name, data_type, is_nullable, column_default, column_key
FROM information_schema.columns
WHERE table_schema = %s AND table_name = 'products'
ORDER BY ordinal_position
"""
execute_sql(query, (MYSQL_DATABASE,))

### Inspect table constraints

Query MySQL metadata to list primary keys, foreign keys, unique rules, and checks.

In [ ]:
query = f"""
SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_schema = %s
ORDER BY table_name, constraint_type, constraint_name
"""
execute_sql(query, (MYSQL_DATABASE,))

### Inspect indexes

The `statistics` metadata view shows the indexes and their ordered columns.

In [ ]:
query = f"""
SELECT table_name, index_name, non_unique, column_name, seq_in_index
FROM information_schema.statistics
WHERE table_schema = %s
ORDER BY table_name, index_name, seq_in_index
"""
execute_sql(query, (MYSQL_DATABASE,))

## Basic performance concepts

An index is an extra lookup structure that helps MySQL find rows without inspecting every row. A full table scan reads the table and may be fine for small tables or queries returning most rows. InnoDB stores table rows using the primary-key B-tree; this is commonly described as a **clustered index**. Other indexes are **secondary** (often called non-clustered conceptually) and contain their indexed values plus the primary key.

Indexes improve suitable reads but cost storage and add work to `INSERT`, `UPDATE`, and `DELETE`. Too many or redundant indexes can therefore hurt performance. Index columns used often in filters, joins, and ordering, then confirm with `EXPLAIN` and realistic data. Our tiny dataset is for learning only—the optimizer may reasonably choose a scan.

### Inspect the query plan

`EXPLAIN` shows how MySQL intends to access the table. Compare the plan with the later indexed version.

In [ ]:
query = f"""
EXPLAIN SELECT order_id, order_date, status
FROM orders
WHERE status = 'PAID'
"""
execute_sql(query)

### Create the `idx_orders_status` index

The index gives MySQL another access path for queries that filter by the indexed column.

In [ ]:
query = f"""
CREATE INDEX idx_orders_status ON orders(status)
"""
execute_sql(query)

### Inspect the query plan

`EXPLAIN` shows how MySQL intends to access the table. Compare the plan with the later indexed version.

In [ ]:
query = f"""
EXPLAIN SELECT order_id, order_date, status
FROM orders
WHERE status = 'PAID'
"""
execute_sql(query)

print("Look at possible_keys, key, rows, and Extra. Small tables may still use a full scan.")

## Review and practice

You have now created a database and tables; used rows, columns, constraints, and three relationship types; queried joins; created a view, function, procedure, and trigger; compared cursors with set-based SQL; used a temporary table; inspected metadata; and introduced indexes and `EXPLAIN`.

Try these exercises:

1. Add a product safely with placeholders and verify its `AUTO_INCREMENT` ID.
2. Attempt a duplicate SKU and observe the `UNIQUE` constraint error.
3. Add an order for customer 3 and two items for that order.
4. Query `order_summary` and calculate 5% GST with `calculate_gst`.
5. Create an index on `orders(customer_id, order_date)` and inspect it in `information_schema.statistics`.
6. Explain why deleting customer 1 fails unless their orders are handled, while deleting a customer profile does not delete its customer.

In [ ]:
# Always close resources when finished. Re-run the connection cell to reconnect.
if connection.is_connected():
    connection.close()
print("Connection closed.")